# ResNet50 — Dog Skin Disease Classifier (Colab Version)
**AI-Based Dog Health Monitoring — Team 193 · PES University**

Colab-ready version of `resnet50.ipynb`. Same 3-stage training, paths updated for Google Drive.

### Before running:
1. Upload your `processed_image/` folder to Google Drive inside a folder called `capstone_data`
2. Enable GPU: **Runtime → Change runtime type → T4 GPU**
3. Run all cells top to bottom

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Step 2 — Configuration

Edit `BASE_DIR` if your folder on Drive has a different name.

In [ ]:
import os

BASE_DIR   = '/content/drive/MyDrive/capstone_data'
TRAIN_DIR  = os.path.join(BASE_DIR, 'processed_image', 'train')
VAL_DIR    = os.path.join(BASE_DIR, 'processed_image', 'valid')
TEST_DIR   = os.path.join(BASE_DIR, 'processed_image', 'test')
MODEL_SAVE = os.path.join(BASE_DIR, 'best_model.keras')  # saved to Drive so it persists

# Verify directories exist
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if os.path.isdir(d):
        print(f'OK  {d}')
    else:
        print(f'MISSING  {d}  ← check your Drive folder structure')

## Step 3 — Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint

import numpy as np
import matplotlib.pyplot as plt

!pip install -q seaborn
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Step 4 — Load Data

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_data = tf.keras.preprocessing.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
test_data = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_data.class_names
print('Classes:', class_names)

In [ ]:
# Cache + prefetch for faster training
AUTOTUNE   = tf.data.AUTOTUNE
train_data = train_data.cache().prefetch(buffer_size=AUTOTUNE)
val_data   = val_data.cache().prefetch(buffer_size=AUTOTUNE)
test_data  = test_data.prefetch(buffer_size=AUTOTUNE)

## Stage 1 — Frozen Base (10 epochs)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

base_model = ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('Stage 1: frozen base, 10 epochs')
history1 = model.fit(train_data, validation_data=val_data, epochs=10)

In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print(f'Stage 1 Test Accuracy: {test_acc:.4f}')

## Stage 2 — Unfreeze Last 30 Layers (5 epochs)

In [ ]:
base_model = model.layers[1]
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('Stage 2: last 30 layers unfrozen, lr=1e-5, 5 epochs')
history2 = model.fit(train_data, validation_data=val_data, epochs=5)

## Stage 3 — ModelCheckpoint (10 epochs) → saves best_model.keras

In [ ]:
checkpoint = ModelCheckpoint(
    MODEL_SAVE,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print(f'Stage 3: ModelCheckpoint → {MODEL_SAVE}')
history3 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[checkpoint]
)

print(f'\nbest_model.keras saved to Drive: {MODEL_SAVE}')

## Final Evaluation

In [ ]:
# Load the best checkpoint for evaluation
best_model = tf.keras.models.load_model(MODEL_SAVE)

test_loss, test_acc = best_model.evaluate(test_data)
print(f'Final Test Accuracy (best checkpoint): {test_acc:.4f}')

In [ ]:
y_true = []
y_pred = []

for images, labels in test_data:
    preds = best_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(
    y_true, y_pred,
    target_names=class_names,
    labels=list(range(len(class_names))),
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('ResNet50 — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Training accuracy curve across all 3 stages
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy'] + history3.history['val_accuracy']

plt.figure(figsize=(10, 4))
plt.plot(all_val_acc, marker='o', linewidth=2)
plt.axvline(9.5,  linestyle='--', color='gray', label='Stage 1→2')
plt.axvline(14.5, linestyle='--', color='orange', label='Stage 2→3')
plt.xlabel('Epoch')
plt.ylabel('Val Accuracy')
plt.title('ResNet50 — Validation Accuracy (all 3 stages)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'\nbest_model.keras is saved at: {MODEL_SAVE}')
print('Use this path in sbert_xgboost.ipynb → IMAGE_MODEL_PATH')